In [1]:
import numpy as np
import torch
import sys
import numpy as np
from sgdml.train import GDMLTrain
from sgdml.predict import GDMLPredict

from molecular.load_other_formats.npz import load_npz
from molecule_data.data import save_geometries

torch.set_default_tensor_type(torch.DoubleTensor) 

In [2]:
rng = np.random.default_rng(42)

n_test = 100
test_inds = rng.permutation(n_test)

coords = load_npz('/home/mholzenkam/dev/datasets/md17/md17_aspirin.npz', key='R')[test_inds]
n_atoms = coords.shape[1]
coords = coords.reshape(n_test, 3*n_atoms)
charges = load_npz('/home/mholzenkam/dev/datasets/md17/md17_aspirin.npz', key='z')
energies = load_npz('/home/mholzenkam/dev/datasets/md17/md17_aspirin.npz', key='E')[test_inds]
forces = load_npz('/home/mholzenkam/dev/datasets/md17/md17_aspirin.npz', key='F')[test_inds]

### Predict

In [3]:
model = np.load('../models/sgdml/md17_aspirin_1k.npz')
gdml = GDMLPredict(model)
e,f = gdml.predict(coords)

mae = abs(e - energies.squeeze()).mean()
print(f'MAE: {mae/23.0621}eV = {mae}kcal/mol')

MAE: 0.012509304537481303eV = 0.2884908321738476kcal/mol


### Train

In [4]:
# save_geometries('test.xyz', coords, charges, energy=energies, forces=forces)

In [15]:
energies.shape

(100, 1)

In [14]:
energies.un[:].shape

(100, 1)

In [5]:
dataset = np.load('test.npz')
n_train = 100

gdml_train = GDMLTrain()
task = gdml_train.create_task(
    dataset, 
    n_train,
    valid_dataset=dataset,
    n_valid=0,
    sig=20, 
    lam=1e-10
)


model = gdml_train.train(task)

np.savez('test_model.npz', **model)

[INFO] Using analytic solver (expected memory use: ~909 MB)


In [7]:
model_loaded = np.load('test_model.npz')

gdml = GDMLPredict(model_loaded)
e,f = gdml.predict(coords)

mae = abs(e - energies.squeeze()).mean()
print(f'MAE: {mae/23.0621}eV = {mae}kcal/mol')

MAE: 0.009724392998549278eV = 0.22426492377184332kcal/mol


In [57]:
# dataset = np.load('../datasets/md17/sgdml_datasets/md17_aspirin.npz')

# Debug

In [25]:
torch.load('results/prediction/refs_e_test') + torch.load('results/data/prepared_data/offsets.pth')['rmd17']

tensor([-6306.1018, -6306.1678, -6305.8710,  ..., -6306.0386, -6306.0719,
        -6305.9747])

In [29]:
torch.load('results/prediction/preds_e_test') + torch.load('results/data/prepared_data/offsets.pth')['rmd17']

tensor([-6306.1017, -6306.1677, -6305.8709,  ..., -6306.0388, -6306.0718,
        -6305.9745])

In [55]:
print(max(abs(torch.load('results/prediction/preds_e_al') - torch.load('results/prediction/refs_e_al'))))
print(min(abs(torch.load('results/prediction/preds_e_al') - torch.load('results/prediction/refs_e_al'))))
print(torch.std(torch.load('results/prediction/preds_e_al') - torch.load('results/prediction/refs_e_al')))

tensor(0.0040)
tensor(3.4129e-09)
tensor(0.0003)


In [56]:
print(max(torch.load('results/prediction/uncertainties_e_al')))
print(min(torch.load('results/prediction/uncertainties_e_al')))
print(torch.std(torch.load('results/prediction/uncertainties_e_al')))

tensor(0.0085)
tensor(3.3424e-10)
tensor(0.0003)


In [69]:
print(max(abs(torch.load('results/prediction/preds_f_al') - torch.load('results/prediction/refs_f_al')).mean(dim=2).mean(dim=1)))
print(min(abs(torch.load('results/prediction/preds_f_al') - torch.load('results/prediction/refs_f_al')).mean(dim=2).mean(dim=1)))
print(torch.std(torch.load('results/prediction/preds_f_al') - torch.load('results/prediction/refs_f_al')))

tensor(0.0111)
tensor(0.0003)
tensor(0.0025)


In [68]:
print(max(torch.load('results/prediction/uncertainties_f_al').mean(dim=2).mean(dim=1)))
print(min(torch.load('results/prediction/uncertainties_f_al').mean(dim=2).mean(dim=1)))
print(torch.std(torch.load('results/prediction/uncertainties_f_al')))

tensor(0.1097)
tensor(0.0005)
tensor(0.0108)


In [64]:
max(abs(torch.load('results/prediction/preds_f_al') - torch.load('results/prediction/refs_f_al')).flatten())

tensor(0.0945)

In [63]:
torch.load('results/prediction/uncertainties_f_al').shape

torch.Size([97000, 12, 3])

In [74]:

test_tensor = torch.randn(10, 4, 3)

torch.sqrt(test_tensor**2).mean(dim=2)

tensor([[0.5334, 1.4712, 0.3850, 1.9302],
        [0.5722, 0.8494, 0.8964, 1.2267],
        [0.4216, 0.2910, 0.5155, 0.6604],
        [0.3122, 0.5941, 1.0028, 0.3897],
        [0.8020, 0.6642, 0.8595, 0.8334],
        [0.6446, 0.8099, 0.3624, 0.3813],
        [1.1554, 1.2537, 0.7279, 0.6006],
        [1.1216, 0.4071, 0.8801, 1.1656],
        [0.7227, 0.9451, 0.7856, 1.1302],
        [0.6827, 0.8544, 0.7763, 0.7302]])

In [75]:
abs(test_tensor).mean(dim=2)

tensor([[0.5334, 1.4712, 0.3850, 1.9302],
        [0.5722, 0.8494, 0.8964, 1.2267],
        [0.4216, 0.2910, 0.5155, 0.6604],
        [0.3122, 0.5941, 1.0028, 0.3897],
        [0.8020, 0.6642, 0.8595, 0.8334],
        [0.6446, 0.8099, 0.3624, 0.3813],
        [1.1554, 1.2537, 0.7279, 0.6006],
        [1.1216, 0.4071, 0.8801, 1.1656],
        [0.7227, 0.9451, 0.7856, 1.1302],
        [0.6827, 0.8544, 0.7763, 0.7302]])